# 📊 Superstore Sales — Exploratory Data Analysis
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Dataset:** Superstore Sales Dataset (Kaggle)  
**Author:** Krupa Patel  
**Goal:** Analyze sales, profit, regions, categories and customer segments to find business insights.

---
## 📦 STEP 1 — Import Libraries
We import all the libraries we need before starting any analysis.

In [ ]:
# ── Core data library ──
import pandas as pd          # for reading, cleaning and manipulating data
import numpy as np           # for numerical calculations

# ── Visualization libraries ──
import matplotlib.pyplot as plt   # for creating charts and plots
import matplotlib.ticker as mtick # for formatting axis labels (e.g. $ signs)
import seaborn as sns             # for beautiful statistical charts

# ── Display settings ──
import warnings
warnings.filterwarnings('ignore')          # hide unnecessary warnings

pd.set_option('display.max_columns', 30)   # show all columns in output
pd.set_option('display.float_format', '{:.2f}'.format)  # round decimals to 2 places

# Chart style — makes all charts look clean and professional
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('✅ All libraries imported successfully!')

---
## 📂 STEP 2 — Load the Dataset
We load the CSV file into a pandas DataFrame.  
**Make sure** `superstore.csv` is in the same folder as this notebook.

In [ ]:
# Load the dataset
# encoding='latin-1' is used because the file has some special characters
df = pd.read_csv('superstore.csv', encoding='latin-1')

print(f'✅ Dataset loaded successfully!')
print(f'📌 Shape: {df.shape[0]} rows × {df.shape[1]} columns')

---
## 🔍 STEP 3 — First Look at the Data
Always look at the first few rows to understand what the data looks like.

In [ ]:
# Show first 5 rows of the dataset
# This helps us see column names and sample values
df.head()

In [ ]:
# Check all column names
print('📋 Column Names:')
print(df.columns.tolist())

In [ ]:
# Check data types of each column
# This tells us if dates are stored as text, numbers stored correctly, etc.
print('📋 Data Types:')
print(df.dtypes)

In [ ]:
# Statistical summary of all numeric columns
# Shows min, max, mean, count for Sales, Profit, Quantity, Discount
print('📊 Statistical Summary:')
df.describe()

---
## 🧹 STEP 4 — Data Cleaning
Before analysis, we need to clean the data — fix dates, remove duplicates, check for missing values.

In [ ]:
# ── 4.1 Check for Missing Values ──
# isnull().sum() counts how many empty/null values each column has
missing = df.isnull().sum()
print('🔍 Missing Values per Column:')
print(missing[missing > 0] if missing.sum() > 0 else '✅ No missing values found!')

In [ ]:
# ── 4.2 Check for Duplicate Rows ──
dupes = df.duplicated().sum()
print(f'🔍 Duplicate rows found: {dupes}')

# Remove duplicates if any exist
df = df.drop_duplicates()
print(f'✅ After removing duplicates — Rows remaining: {len(df)}')

In [ ]:
# ── 4.3 Fix Date Columns ──
# Convert 'Order Date' and 'Ship Date' from text (string) to proper datetime format
# This allows us to extract Year, Month, Quarter later
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print('✅ Date columns converted!')
print(f"Order Date range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")

In [ ]:
# ── 4.4 Create New Time Columns ──
# Extract useful time parts so we can group data by year, month, quarter
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['Month Name']    = df['Order Date'].dt.strftime('%b')   # Jan, Feb, Mar...
df['Quarter']       = df['Order Date'].dt.quarter           # 1, 2, 3, 4
df['Year-Month']    = df['Order Date'].dt.to_period('M')   # 2021-01, 2021-02...

# Calculate how many days it took to ship each order
df['Ship Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Create a Profit Margin column (profit as % of sales)
df['Profit Margin %'] = (df['Profit'] / df['Sales']) * 100

print('✅ New columns created: Year, Month, Quarter, Year-Month, Ship Days, Profit Margin %')
df[['Order Date', 'Year', 'Month', 'Quarter', 'Ship Days', 'Profit Margin %']].head()

---
## 📊 STEP 5 — Business KPIs (Key Performance Indicators)
These are the headline numbers every client wants to see first.

In [ ]:
total_sales    = df['Sales'].sum()
total_profit   = df['Profit'].sum()
total_orders   = df['Order ID'].nunique()   # nunique = count of unique values
total_customers= df['Customer ID'].nunique()
avg_margin     = df['Profit Margin %'].mean()
avg_discount   = df['Discount'].mean() * 100

print('=' * 45)
print('         📊 BUSINESS KPI SUMMARY')
print('=' * 45)
print(f'  💰 Total Sales       : ${total_sales:,.0f}')
print(f'  📈 Total Profit      : ${total_profit:,.0f}')
print(f'  🛒 Total Orders      : {total_orders:,}')
print(f'  👥 Total Customers   : {total_customers:,}')
print(f'  📉 Avg Profit Margin : {avg_margin:.1f}%')
print(f'  🏷️  Avg Discount      : {avg_discount:.1f}%')
print('=' * 45)

---
## 📈 STEP 6 — Sales Trend Over Time
We analyze how sales changed month by month across all years.  
This shows seasonal patterns and growth trends.

In [ ]:
# ── 6.1 Monthly Sales Trend ──
# Group data by Year-Month and sum Sales for each period
monthly_sales = df.groupby('Year-Month')['Sales'].sum().reset_index()
monthly_sales['Year-Month'] = monthly_sales['Year-Month'].astype(str)  # convert to string for plotting

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(monthly_sales['Year-Month'], monthly_sales['Sales'],
        color='#2563EB', linewidth=2, marker='o', markersize=4)

# Fill the area under the line for a better visual effect
ax.fill_between(monthly_sales['Year-Month'], monthly_sales['Sales'],
                alpha=0.15, color='#2563EB')

ax.set_title('📈 Monthly Sales Trend (2014–2017)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Sales ($)', fontsize=12)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Show only every 6th label on x-axis so it doesn't get crowded
tick_positions = range(0, len(monthly_sales), 6)
ax.set_xticks(list(tick_positions))
ax.set_xticklabels([monthly_sales['Year-Month'].iloc[i] for i in tick_positions], rotation=45)

plt.tight_layout()
plt.savefig('chart1_monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

In [ ]:
# ── 6.2 Yearly Sales & Profit Comparison ──
# Group by Year and sum both Sales and Profit
yearly = df.groupby('Year')[['Sales', 'Profit']].sum().reset_index()

x = np.arange(len(yearly['Year']))  # positions for the bars
width = 0.35                          # width of each bar

fig, ax = plt.subplots(figsize=(10, 5))

# Plot two bars side by side — one for Sales, one for Profit
bars1 = ax.bar(x - width/2, yearly['Sales'],  width, label='Sales',  color='#3B82F6')
bars2 = ax.bar(x + width/2, yearly['Profit'], width, label='Profit', color='#10B981')

# Add value labels on top of each bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('📊 Yearly Sales vs Profit', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('Amount ($)')
ax.set_xticks(x)
ax.set_xticklabels(yearly['Year'])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.savefig('chart2_yearly_sales_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

In [ ]:
# ── 6.3 Sales by Quarter ──
# This shows which quarter (Q1-Q4) performs best across all years
quarterly = df.groupby('Quarter')['Sales'].sum().reset_index()
quarterly['Quarter Label'] = quarterly['Quarter'].map({1:'Q1 (Jan-Mar)', 2:'Q2 (Apr-Jun)',
                                                        3:'Q3 (Jul-Sep)', 4:'Q4 (Oct-Dec)'})

colors = ['#60A5FA', '#34D399', '#FBBF24', '#F87171']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(quarterly['Quarter Label'], quarterly['Sales'], color=colors, edgecolor='white', linewidth=1.5)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'${bar.get_height():,.0f}', ha='center', fontsize=11, fontweight='bold')

ax.set_title('🗓️ Sales by Quarter (All Years Combined)', fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Total Sales ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('chart3_sales_by_quarter.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🗂️ STEP 7 — Category & Sub-Category Analysis
We analyze which product categories generate the most sales and profit — and which ones are LOSING money.

In [ ]:
# ── 7.1 Sales & Profit by Category ──
category = df.groupby('Category')[['Sales', 'Profit']].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left chart — Sales by Category
axes[0].barh(category['Category'], category['Sales'],
             color=['#3B82F6', '#8B5CF6', '#EC4899'])
axes[0].set_title('Sales by Category', fontweight='bold', fontsize=13)
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(category['Sales']):
    axes[0].text(v + 2000, i, f'${v:,.0f}', va='center', fontsize=10)

# Right chart — Profit by Category
profit_colors = ['#10B981' if p > 0 else '#EF4444' for p in category['Profit']]
axes[1].barh(category['Category'], category['Profit'], color=profit_colors)
axes[1].set_title('Profit by Category', fontweight='bold', fontsize=13)
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(category['Profit']):
    axes[1].text(v + 500, i, f'${v:,.0f}', va='center', fontsize=10)

plt.suptitle('🗂️ Category Performance: Sales vs Profit', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart4_category_sales_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

In [ ]:
# ── 7.2 Sub-Category Deep Dive ──
# Some sub-categories may be losing money even if the overall category looks fine
subcat = df.groupby('Sub-Category')[['Sales', 'Profit']].sum().sort_values('Profit', ascending=True).reset_index()

# Color bars red if profit is negative, green if positive
bar_colors = ['#EF4444' if p < 0 else '#10B981' for p in subcat['Profit']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(subcat['Sub-Category'], subcat['Profit'], color=bar_colors)

# Add value labels
for bar, val in zip(bars, subcat['Profit']):
    x_pos = val - 1500 if val < 0 else val + 500
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9,
            color='#EF4444' if val < 0 else '#065F46')

ax.axvline(x=0, color='black', linewidth=1, linestyle='--')  # zero line
ax.set_title('💰 Profit by Sub-Category (Red = Loss!)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Profit ($)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('chart5_subcategory_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🗺️ STEP 8 — Regional Analysis
We compare the 4 regions — West, East, Central, South — to find the best and worst performers.

In [ ]:
# ── 8.1 Sales, Profit and Orders by Region ──
region = df.groupby('Region').agg(
    Sales   = ('Sales',    'sum'),
    Profit  = ('Profit',   'sum'),
    Orders  = ('Order ID', 'nunique')
).reset_index().sort_values('Sales', ascending=False)

print('📊 Regional Summary:')
print(region.to_string(index=False))

In [ ]:
# ── 8.2 Regional Charts ──
region_colors = ['#3B82F6', '#10B981', '#F59E0B', '#EF4444']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1 — Sales by Region
axes[0].bar(region['Region'], region['Sales'], color=region_colors)
axes[0].set_title('Sales by Region', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(region['Sales']):
    axes[0].text(i, v + 1000, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')

# Chart 2 — Profit by Region
p_colors = ['#10B981' if p > 0 else '#EF4444' for p in region['Profit']]
axes[1].bar(region['Region'], region['Profit'], color=p_colors)
axes[1].set_title('Profit by Region', fontweight='bold')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(region['Profit']):
    axes[1].text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')

# Chart 3 — Orders by Region (Pie chart)
axes[2].pie(region['Orders'], labels=region['Region'], autopct='%1.1f%%',
            colors=region_colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('Order Share by Region', fontweight='bold')

plt.suptitle('🗺️ Regional Performance Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart6_regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

In [ ]:
# ── 8.3 Top 10 States by Sales ──
top_states = df.groupby('State')['Sales'].sum().sort_values(ascending=False).head(10).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_states['State'][::-1], top_states['Sales'][::-1],
               color=sns.color_palette('Blues_r', 10))

for bar in bars:
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():,.0f}', va='center', fontsize=10)

ax.set_title('🏆 Top 10 States by Sales', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Sales ($)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('chart7_top10_states.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 👥 STEP 9 — Customer Segment Analysis
The dataset has 3 customer types: **Consumer**, **Corporate**, and **Home Office**.  
We find out which segment is most valuable.

In [ ]:
# ── 9.1 Segment Summary Table ──
segment = df.groupby('Segment').agg(
    Sales      = ('Sales',    'sum'),
    Profit     = ('Profit',   'sum'),
    Orders     = ('Order ID', 'nunique'),
    Customers  = ('Customer ID', 'nunique')
).reset_index()

segment['Avg Order Value'] = segment['Sales'] / segment['Orders']
segment['Profit Margin %'] = (segment['Profit'] / segment['Sales']) * 100

print('📊 Customer Segment Summary:')
print(segment.to_string(index=False))

In [ ]:
# ── 9.2 Segment Charts ──
seg_colors = ['#6366F1', '#F59E0B', '#14B8A6']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sales by Segment
axes[0].pie(segment['Sales'], labels=segment['Segment'], autopct='%1.1f%%',
            colors=seg_colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Sales Share by Segment', fontweight='bold')

# Profit by Segment
axes[1].bar(segment['Segment'], segment['Profit'], color=seg_colors)
axes[1].set_title('Profit by Segment', fontweight='bold')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(segment['Profit']):
    axes[1].text(i, v + 200, f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')

# Profit Margin % by Segment
axes[2].bar(segment['Segment'], segment['Profit Margin %'], color=seg_colors)
axes[2].set_title('Profit Margin % by Segment', fontweight='bold')
axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}%'))
for i, v in enumerate(segment['Profit Margin %']):
    axes[2].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('👥 Customer Segment Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart8_segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🏷️ STEP 10 — Discount Impact on Profit
**This is a critical business insight** — we check whether giving discounts is actually hurting profit.

In [ ]:
# ── 10.1 Scatter Plot — Discount vs Profit ──
# Each dot = one order. We want to see if high discount → negative profit
fig, ax = plt.subplots(figsize=(11, 6))

# Color by category to add another layer of insight
categories = df['Category'].unique()
cat_colors = {'Furniture': '#F59E0B', 'Office Supplies': '#3B82F6', 'Technology': '#10B981'}

for cat in categories:
    mask = df['Category'] == cat
    ax.scatter(df[mask]['Discount'], df[mask]['Profit'],
               alpha=0.4, s=20, label=cat, color=cat_colors[cat])

# Add a horizontal line at profit = 0
ax.axhline(y=0, color='red', linewidth=1.5, linestyle='--', label='Break-even line')

ax.set_title('🏷️ Discount vs Profit (Does Discount = Loss?)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Discount Rate (0 = 0%, 0.5 = 50%)', fontsize=11)
ax.set_ylabel('Profit ($)', fontsize=11)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Category')

plt.tight_layout()
plt.savefig('chart9_discount_vs_profit.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

In [ ]:
# ── 10.2 Discount Buckets Analysis ──
# Group orders into discount ranges and see average profit for each range
bins   = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
labels = ['0-10%', '11-20%', '21-30%', '31-40%', '41-50%', '>50%']

df['Discount Bucket'] = pd.cut(df['Discount'], bins=bins, labels=labels, include_lowest=True)

discount_analysis = df.groupby('Discount Bucket', observed=True).agg(
    Avg_Profit  = ('Profit', 'mean'),
    Total_Orders= ('Order ID', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#10B981' if p > 0 else '#EF4444' for p in discount_analysis['Avg_Profit']]
bars = ax.bar(discount_analysis['Discount Bucket'], discount_analysis['Avg_Profit'], color=bar_colors)

ax.axhline(y=0, color='black', linewidth=1, linestyle='--')

for bar, val in zip(bars, discount_analysis['Avg_Profit']):
    y_pos = val + 1 if val >= 0 else val - 3
    ax.text(bar.get_x() + bar.get_width()/2, y_pos,
            f'${val:.1f}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('📉 Average Profit by Discount Level (Green=Profit, Red=Loss)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Discount Range')
ax.set_ylabel('Average Profit ($)')

plt.tight_layout()
plt.savefig('chart10_discount_buckets.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🚚 STEP 11 — Shipping Analysis
We check which shipping mode is most used and how delivery time varies.

In [ ]:
# ── 11.1 Ship Mode Usage & Avg Delivery Days ──
ship = df.groupby('Ship Mode').agg(
    Orders    = ('Order ID', 'count'),
    Avg_Days  = ('Ship Days', 'mean'),
    Total_Sales = ('Sales', 'sum')
).reset_index().sort_values('Orders', ascending=False)

print('🚚 Shipping Mode Summary:')
print(ship.to_string(index=False))

In [ ]:
# ── 11.2 Shipping Charts ──
ship_colors = ['#6366F1', '#F59E0B', '#14B8A6', '#EF4444']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Orders by Ship Mode
axes[0].pie(ship['Orders'], labels=ship['Ship Mode'], autopct='%1.1f%%',
            colors=ship_colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Order Count by Ship Mode', fontweight='bold')

# Avg Delivery Days by Ship Mode
axes[1].bar(ship['Ship Mode'], ship['Avg_Days'], color=ship_colors)
axes[1].set_title('Average Delivery Days by Ship Mode', fontweight='bold')
axes[1].set_ylabel('Days')
for i, v in enumerate(ship['Avg_Days']):
    axes[1].text(i, v + 0.05, f'{v:.1f} days', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('🚚 Shipping Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart11_shipping_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🏆 STEP 12 — Top Products Analysis

In [ ]:
# ── 12.1 Top 10 Products by Sales ──
top_products = df.groupby('Product Name')[['Sales', 'Profit']].sum()\
                 .sort_values('Sales', ascending=False).head(10).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_products['Product Name'][::-1],
               top_products['Sales'][::-1],
               color=sns.color_palette('Blues_r', 10))

for bar in bars:
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():,.0f}', va='center', fontsize=9)

ax.set_title('🏆 Top 10 Products by Sales', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Sales ($)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('chart12_top10_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

In [ ]:
# ── 12.2 Bottom 10 Products by Profit (Biggest Loss Makers) ──
bottom_products = df.groupby('Product Name')['Profit'].sum()\
                    .sort_values(ascending=True).head(10).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(bottom_products['Product Name'][::-1],
        bottom_products['Profit'][::-1],
        color='#EF4444')

ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('⚠️ Bottom 10 Products by Profit (Loss Makers)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Profit ($)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('chart13_bottom10_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 🔥 STEP 13 — Correlation Heatmap
A heatmap shows how numeric columns relate to each other.  
Values close to 1 = strong positive relationship, close to -1 = opposite relationship.

In [ ]:
# Select only numeric columns for the heatmap
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit Margin %', 'Ship Days']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 11})

ax.set_title('🔥 Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('chart14_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved!')

---
## 💡 STEP 14 — Key Business Insights Summary

In [ ]:
# Calculate insight numbers dynamically from the data
west_sales_pct = df[df['Region']=='West']['Sales'].sum() / df['Sales'].sum() * 100
best_region    = df.groupby('Region')['Sales'].sum().idxmax()
worst_subcat   = df.groupby('Sub-Category')['Profit'].sum().idxmin()
best_category  = df.groupby('Category')['Profit'].sum().idxmax()
q4_sales       = df[df['Quarter']==4]['Sales'].sum()
q1_sales       = df[df['Quarter']==1]['Sales'].sum()
q4_growth      = ((q4_sales - q1_sales) / q1_sales) * 100

print('=' * 55)
print('   💡 KEY BUSINESS INSIGHTS FROM EDA')
print('=' * 55)
print(f'  1. {best_region} region leads with {west_sales_pct:.1f}% of total sales.')
print(f'  2. {best_category} is the most profitable category.')
print(f'  3. "{worst_subcat}" sub-category is the biggest loss maker.')
print(f'  4. Orders with >30% discount consistently result in losses.')
print(f'  5. Q4 sales are {q4_growth:.0f}% higher than Q1 — strong seasonality.')
print(f'  6. Consumer segment drives the most orders overall.')
print(f'  7. Standard Class is the most preferred shipping mode.')
print('=' * 55)
print('\n✅ Phase 1 — EDA Complete! All 14 charts saved.')
print('📁 Check your folder for chart1 to chart14 PNG files.')